In [1]:
import json
import re
import os
import sys
import ast
import math
from pint import UnitRegistry
from sympy import parse_expr

# 导入路径配置
current_dir = os.path.dirname(os.path.abspath(r"")) 
while not os.path.exists(os.path.join(current_dir, "requirements.txt")):
    current_dir = os.path.dirname(current_dir)
project_root = current_dir
sys.path.append(project_root)

In [2]:
# 初始化单位注册表
ureg = UnitRegistry()
geometry_symbol_table = {}

unit_lr = 1e-3 * ureg.meter

mm = ureg.millimeter
cm = ureg.centimeter
m = ureg.meter

a = ((1.5*mm).to_base_units()).magnitude
print(a)
print((1.6 * cm)/unit_lr)
print((1.7 * m)/unit_lr)
print(round(((1.6 * cm).to(unit_lr) / unit_lr).magnitude, 8))

0.0015
1600.0 centimeter / meter
1700.0 dimensionless
16.0


In [15]:
import re
float_ext_exp = r"[+-]?([0-9]+([.][0-9]*)?([eE][+-]?[0-9]+)?|[.][0-9]+([eE][+-]?[0-9]+)?)[_A-Za-z]*"

def ensure_units(values: list[str], default_unit="meter") -> list[str]:
    """
    确保数值列表中每个元素都带单位。
    - 已经有单位的保持不变 (例如 '0.5*mm')
    - 没有单位的自动补上 '*{default_unit}'
    """
    processed = []
    for val in values:
        val = val.strip()
        if "*" in val:  # 已经带单位
            processed.append(val)
        else:
            try:
                # 判断是否是纯数值
                _ = float(val)
                processed.append(f"{val}*{default_unit}")
            except ValueError:
                # 不是数字，直接保留
                processed.append(val)
    return processed

def process_line_command(cmd: str, default_unit="meter") -> str:
    """
    处理 LINE 命令，保证最后四个分量有单位。
    输入：原始命令字符串
    输出：处理后的命令字符串
    """
    # ---- 预清理 ----
    clean_cmd = cmd.replace(",", " ")
    clean_cmd = re.sub(r"\s*\*\s*", "*", clean_cmd)

    tokens = clean_cmd.split()
    if len(tokens) >= 5:  # LINE name ... x1 y1 x2 y2
        coords = tokens[-4:]
        coords = ensure_units(coords, default_unit=default_unit)
        tokens = tokens[:-4] + coords
    return " ".join(tokens)


def process_area_command(cmd: str, default_unit="meter") -> str:
    """
    处理 AREA 命令，保证所有数值分量有单位。
    输入：原始命令字符串
    输出：处理后的命令字符串
    """
    # ---- 预清理 ----
    clean_cmd = cmd.replace(",", " ")
    clean_cmd = re.sub(r"\s*\*\s*", "*", clean_cmd)

    tokens = clean_cmd.split()
    coords = [t for t in tokens if re.match(float_ext_exp, t)]
    coords = ensure_units(coords, default_unit=default_unit)

    # 替换 tokens 中的数值
    idx = 0
    for i, t in enumerate(tokens):
        if re.match(float_ext_exp, t):
            tokens[i] = coords[idx]
            idx += 1
    return " ".join(tokens)

cmd1 = "LINE INLET CONFORMAL 0.0    0.0575 0.0 0.143 ;"
cmd2 = "AREA EMITTER CONFORMAL 441.6   50  , 764.8 , 80 ;"
print(process_line_command(cmd1))
print(process_area_command(cmd2))

LINE INLET CONFORMAL 0.0 0.0575*meter 0.0*meter 0.143*meter ;
AREA EMITTER CONFORMAL 441.6*meter 50*meter 764.8*meter 80*meter ;


In [2]:
## 将（X，Y）转换为（0，Y，X）
input = [(140.1, 0.0), (12.0, 0.0), (12.0, 11.4), (17.0, 11.4), (17.0, 11.8), (12.0, 11.8), (0.0, 11.8), (0.0, 22.0), (22.0, 22.0), (30.0, 13.0), (35.0, 13.0), (35.0, 19.0), (43.7, 19.0), (43.7, 13.0), (47.0, 13.0), (47.0, 14.2), (48.0, 14.2), (50.2, 13.0), (52.0, 13.0), (54.2, 14.2), (55.2, 14.2), (57.4, 13.0), (59.2, 13.0), (61.4, 14.2), (62.4, 14.2), (64.6, 13.0), (66.4, 13.0), (68.6, 14.2), (69.6, 14.2), (71.8, 13.0), (73.6, 13.0), (75.8, 14.2), (76.8, 14.2), (79.0, 13.0), (80.8, 13.0), (83.0, 14.2), (84.0, 14.2), (86.2, 13.0), (88.0, 13.0), (90.2, 14.2), (91.2, 14.2), (93.4, 13.0), (95.2, 13.0), (97.4, 14.2), (98.4, 14.2), (100.6, 13.0), (102.4, 13.0), (104.6, 14.2), (105.6, 14.2), (107.8, 13.0), (109.6, 13.0), (111.8, 14.2), (112.8, 14.2), (112.8, 13.0), (113.8, 13.0), (113.8, 16.4), (115.1, 16.4), (115.1, 12.8), (140.1, 12.8), (140.1, 0.0)]
output = "[  "
for point in input:
    X1, X2 = point[0], point[1]
    output += f"0.0 {X2} {X1}  "
output += "]"
print(output)

[  0.0 0.0 140.1  0.0 0.0 12.0  0.0 11.4 12.0  0.0 11.4 17.0  0.0 11.8 17.0  0.0 11.8 12.0  0.0 11.8 0.0  0.0 22.0 0.0  0.0 22.0 22.0  0.0 13.0 30.0  0.0 13.0 35.0  0.0 19.0 35.0  0.0 19.0 43.7  0.0 13.0 43.7  0.0 13.0 47.0  0.0 14.2 47.0  0.0 14.2 48.0  0.0 13.0 50.2  0.0 13.0 52.0  0.0 14.2 54.2  0.0 14.2 55.2  0.0 13.0 57.4  0.0 13.0 59.2  0.0 14.2 61.4  0.0 14.2 62.4  0.0 13.0 64.6  0.0 13.0 66.4  0.0 14.2 68.6  0.0 14.2 69.6  0.0 13.0 71.8  0.0 13.0 73.6  0.0 14.2 75.8  0.0 14.2 76.8  0.0 13.0 79.0  0.0 13.0 80.8  0.0 14.2 83.0  0.0 14.2 84.0  0.0 13.0 86.2  0.0 13.0 88.0  0.0 14.2 90.2  0.0 14.2 91.2  0.0 13.0 93.4  0.0 13.0 95.2  0.0 14.2 97.4  0.0 14.2 98.4  0.0 13.0 100.6  0.0 13.0 102.4  0.0 14.2 104.6  0.0 14.2 105.6  0.0 13.0 107.8  0.0 13.0 109.6  0.0 14.2 111.8  0.0 14.2 112.8  0.0 13.0 112.8  0.0 13.0 113.8  0.0 16.4 113.8  0.0 16.4 115.1  0.0 12.8 115.1  0.0 12.8 140.1  0.0 0.0 140.1  ]


In [1]:
# 字符串转换为(x,y)元组列表的函数
def convert_to_xy_pairs(input_str):
    # 1. 分割字符串为数字列表
    num_list = list(map(float, input_str[1:-1].split()))
    # 2. 检查长度有效性
    if len(num_list) % 3 != 0:
        raise ValueError("输入的数字总数必须是3的倍数")
    # 3. 转换为(x,y)元组
    return [
        (num_list[i], num_list[i+1]) 
        for i in range(0, len(num_list), 3)
    ]
# 测试用例
input_data1 = "[0.0 0.0 0.0 0.0 11.8 0.0 12.0 11.8 0.0 12.0 0.0 0.0 0.0 0.0 0.0]"
input_data2 = "[12.0 11.4 0.0 12.0 11.8 0.0 17.0 11.8 0.0 17.0 11.4 0.0 12.0 11.4 0.0]"
input_data3 = "[0.0 22.0 0.0 22.0 22.0 0.0 30.0 13.0 0.0 35.0 13.0 0.0 35.0 19.0 0.0 43.7 19.0 0.0 43.7 13.0 0.0 47.0 13.0 0.0 47.0 25.0 0.0 0.0 25.0 0.0 0.0 22.0 0.0]"
input_data4 = "[47.0 14.2 0.0 48.0 14.2 0.0 50.2 13.0 0.0 52.0 13.0 0.0 54.2 14.2 0.0 55.2 14.2 0.0 57.4 13.0 0.0 59.2 13.0 0.0 61.4 14.2 0.0 62.4 14.2 0.0 64.6 13.0 0.0 66.4 13.0 0.0 68.6 14.2 0.0 69.6 14.2 0.0 71.8 13.0 0.0 73.6 13.0 0.0 75.8 14.2 0.0 76.8 14.2 0.0 79.0 13.0 0.0 80.8 13.0 0.0 83.0 14.2 0.0 84.0 14.2 0.0 86.2 13.0 0.0 88.0 13.0 0.0 90.2 14.2 0.0 91.2 14.2 0.0 93.4 13.0 0.0 95.2 13.0 0.0 97.4 14.2 0.0 98.4 14.2 0.0 100.6 13.0 0.0 102.4 13.0 0.0 104.6 14.2 0.0 105.6 14.2 0.0 107.8 13.0 0.0 109.6 13.0 0.0 111.8 14.2 0.0 112.8 14.2 0.0 112.8 13.0 0.0 113.8 13.0 0.0 113.8 16.4 0.0 115.1 16.4 0.0 115.1 18.8 0.0 47.0 18.8 0.0 47.0 14.2 0.0]"
input_data5 = "[115.1 12.8 0.0 115.1 18.8 0.0 140.1 18.8 0.0 140.1 12.8 0.0 115.1 12.8 0.0]"
input_data6 = "[0.0 0.0 0.0 0.0 12.3 0.0 16.95 12.3 0.0 16.95 0.0 0.0 0.0 0.0 0.0]"
  
output1 = convert_to_xy_pairs(input_data1)
output2 = convert_to_xy_pairs(input_data2)  
output3 = convert_to_xy_pairs(input_data3)
output4 = convert_to_xy_pairs(input_data4)
output5 = convert_to_xy_pairs(input_data5)
output6 = convert_to_xy_pairs(input_data6)

output = "[\n"+ \
    str(output1)+",\n"+ \
    str(output2)+",\n"+ \
    str(output3)+",\n"+ \
    str(output4)+",\n"+ \
    str(output5)+",\n"+ \
    str(output6)+"\n"+ \
    "]"
print(output)

[
[(0.0, 0.0), (0.0, 11.8), (12.0, 11.8), (12.0, 0.0), (0.0, 0.0)],
[(12.0, 11.4), (12.0, 11.8), (17.0, 11.8), (17.0, 11.4), (12.0, 11.4)],
[(0.0, 22.0), (22.0, 22.0), (30.0, 13.0), (35.0, 13.0), (35.0, 19.0), (43.7, 19.0), (43.7, 13.0), (47.0, 13.0), (47.0, 25.0), (0.0, 25.0), (0.0, 22.0)],
[(47.0, 14.2), (48.0, 14.2), (50.2, 13.0), (52.0, 13.0), (54.2, 14.2), (55.2, 14.2), (57.4, 13.0), (59.2, 13.0), (61.4, 14.2), (62.4, 14.2), (64.6, 13.0), (66.4, 13.0), (68.6, 14.2), (69.6, 14.2), (71.8, 13.0), (73.6, 13.0), (75.8, 14.2), (76.8, 14.2), (79.0, 13.0), (80.8, 13.0), (83.0, 14.2), (84.0, 14.2), (86.2, 13.0), (88.0, 13.0), (90.2, 14.2), (91.2, 14.2), (93.4, 13.0), (95.2, 13.0), (97.4, 14.2), (98.4, 14.2), (100.6, 13.0), (102.4, 13.0), (104.6, 14.2), (105.6, 14.2), (107.8, 13.0), (109.6, 13.0), (111.8, 14.2), (112.8, 14.2), (112.8, 13.0), (113.8, 13.0), (113.8, 16.4), (115.1, 16.4), (115.1, 18.8), (47.0, 18.8), (47.0, 14.2)],
[(115.1, 12.8), (115.1, 18.8), (140.1, 18.8), (140.1, 12.8), (

In [5]:
# 字符串转换为(x,y)元组列表的函数
def convert_to_xy_pairs(input_str):
    # 1. 分割字符串为数字列表
    num_list = list(map(float, input_str[1:-1].split()))
    # 2. 检查长度有效性
    if len(num_list) % 3 != 0:
        raise ValueError("输入的数字总数必须是3的倍数")
    # 3. 转换为(x,y)元组
    return [
        (num_list[i+2], num_list[i+1]) 
        for i in range(0, len(num_list), 3)
    ]
# 测试用例
input_data1 = "[0.0 69.0 0.0 0.0 69.0 50.0 0.0 70.0 50.0 0.0 70.0 0.0 0.0 69.0 0.0]"
input_data2 = "[0.0 69.0 50.0 0.0 69.0 200.0 0.0 70.0 200.0 0.0 70.0 50.0 0.0 69.0 50.0]"
input_data3 = "[0.0 69.0 200.0 0.0 69.0 250.0 0.0 70.0 250.0 0.0 70.0 200.0 0.0 69.0 200.0]"
input_data4 = "[0.0 90.0 0.0 0.0 90.0 250.0 0.0 91.0 250.0 0.0 91.0 0.0 0.0 90.0 0.0]"
input_data5 = "[0.0 80.0 150.0 0.0 80.0 250.0 0.0 90.0 250.0 0.0 90.0 150.0 0.0 80.0 150.0]"
#input_data6 = "[0.0 69.0 0.0 0.0 69.0 250.0 0.0 91.0 250.0 0.0 91.0 0.0 0.0 69.0 0.0]"
  
output1 = convert_to_xy_pairs(input_data1)
output2 = convert_to_xy_pairs(input_data2)  
output3 = convert_to_xy_pairs(input_data3)
output4 = convert_to_xy_pairs(input_data4)
output5 = convert_to_xy_pairs(input_data5)
#output6 = convert_to_xy_pairs(input_data6)

output = "[\n"+ \
    str(output1)+",\n"+ \
    str(output2)+",\n"+ \
    str(output3)+",\n"+ \
    str(output4)+",\n"+ \
    str(output5)+",\n"+ \
"]"
#    str(output6)+"\n"+ \
    
print(output)

[
[(0.0, 69.0), (50.0, 69.0), (50.0, 70.0), (0.0, 70.0), (0.0, 69.0)],
[(50.0, 69.0), (200.0, 69.0), (200.0, 70.0), (50.0, 70.0), (50.0, 69.0)],
[(200.0, 69.0), (250.0, 69.0), (250.0, 70.0), (200.0, 70.0), (200.0, 69.0)],
[(0.0, 90.0), (250.0, 90.0), (250.0, 91.0), (0.0, 91.0), (0.0, 90.0)],
[(150.0, 80.0), (250.0, 80.0), (250.0, 90.0), (150.0, 90.0), (150.0, 80.0)],
]
